

**Sections**
1. Overall test performance — F2, precision, recall, Brier score, bootstrap 95% CIs
2. Per-park breakdown — F2 heatmaps across six Nigerian national parks
3. Calibration diagnostics — reliability diagrams for all models
4. Lead-time analysis — mean days of advance warning per threat

## Metrics Reference

All metrics computed on the held-out test set (Jan 2024 onward) at threshold 0.50 unless noted.

| Metric | What it measures | Range | Better when |
|---|---|---|---|
| **F2** | Weighted harmonic mean of precision and recall, recall weighted 2× — **primary metric** | 0–1 | Higher |
| **Precision** | Of all positive predictions, fraction correct | 0–1 | Higher |
| **Recall** | Of all actual positives, fraction detected | 0–1 | Higher |
| **Brier** | Mean squared error between predicted probability and true label | 0–1 | Lower |
| **ROC-AUC** | Probability model ranks a positive above a negative; threshold-independent | 0–1 | Higher |
| **95% CI** | Bootstrap CI for F2 (1,000 non-parametric resamples) | — | Narrow |
| **Persist F2** | F2 of persistence baseline: predict today's label as tomorrow's | 0–1 | — |
| **Lead time** | Mean days advance warning before event onset, at P ≥ 0.70 | days | Higher |

**Why F2 over F1?** Missing a real threat (false negative) costs more than a false alarm.
F2 penalises missed detections twice as heavily as F1.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from sklearn.calibration import calibration_curve

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src" / "models"))

from model_utils import apply_platt, load_splits, prep_arrays
from evaluate import THREAT_LABELS
from seq_utils import build_sequence_splits
from supervised.train_lstm import ThreatLSTM, predict_proba as lstm_predict
from supervised.train_transformer import ThreatTransformer, predict_proba as tf_predict

RESULTS = ROOT / "results"
MODELS  = RESULTS / "models"
(RESULTS / "plots").mkdir(parents=True, exist_ok=True)

THREAT_SHORT = {
    "fire_within_30d":       "Fire",
    "drought_within_30d":    "Drought",
    "vegetation_within_30d": "Vegetation",
}
PARK_LABELS = {
    "chad_basin":    "Chad Basin",
    "cross_river":   "Cross River",
    "gashaka_gumti": "Gashaka-Gumti",
    "kainji_lake":   "Kainji Lake",
    "old_oyo":       "Old Oyo",
    "yankari":       "Yankari",
}
MODEL_COLORS = {
    "RF":          "#2196F3",
    "XGBoost":     "#FF5722",
    "LSTM":        "#4CAF50",
    "Transformer": "#9C27B0",
}
MODELS_ORDER = ["RF", "XGBoost", "LSTM", "Transformer"]

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Setup complete. ROOT:", ROOT)

In [ ]:
result_files = {
    "RF":          "rf_supervised_results.json",
    "XGBoost":     "xgb_supervised_results.json",
    "LSTM":        "lstm_supervised_results.json",
    "Transformer": "transformer_supervised_results.json",
}
res = {}
for name, fname in result_files.items():
    with open(RESULTS / fname) as f:
        res[name] = json.load(f)
    print(f"{name:12s} test mean-F2: {res[name]['test_mean_f2']:.4f}")

## 1. Overall Test Performance

F2 (beta=2) is the primary metric. Bootstrap 95% CIs use 1,000 non-parametric resamples.
Neural models (LSTM, Transformer) use early stopping on validation loss; tree models use grid search.

In [ ]:
rows = []
for model_name in MODELS_ORDER:
    for label in THREAT_LABELS:
        m  = res[model_name]["test_metrics"][label]
        ci = m["bootstrap_f2"]
        rows.append({
            "Model":          model_name,
            "Threat":         THREAT_SHORT[label],
            "F2":             round(m["f2"], 4),
            "95% CI":         f"[{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]",
            "Precision":      round(m["precision"], 4),
            "Recall":         round(m["recall"], 4),
            "Brier":          round(m["brier"], 4),
            "ROC-AUC":        round(m["roc_auc"], 4),
            "Persist F2":     round(m["persistence_f2"], 4),
            "Beats Baseline": "Yes" if m["beats_baseline"] else "No",
        })

df_compare = pd.DataFrame(rows).set_index(["Model", "Threat"])
df_compare

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, label in zip(axes, THREAT_LABELS):
    x_pos = np.arange(len(MODELS_ORDER))
    for j, model_name in enumerate(MODELS_ORDER):
        m  = res[model_name]["test_metrics"][label]
        ci = m["bootstrap_f2"]
        f2 = m["f2"]
        lo = f2 - ci["ci_lower"]
        hi = ci["ci_upper"] - f2
        ax.bar(j, f2, color=MODEL_COLORS[model_name], alpha=0.85, width=0.6)
        ax.errorbar(j, f2, yerr=[[lo], [hi]], fmt="none",
                    color="black", capsize=4, linewidth=1.5)
        ax.text(j, ci["ci_upper"] + 0.012, f"{f2:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")

    persist = res["RF"]["test_metrics"][label]["persistence_f2"]
    ax.axhline(persist, color="gray", linestyle="--", linewidth=1)
    ax.set_title(THREAT_SHORT[label], fontweight="bold")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(MODELS_ORDER, rotation=15, ha="right")
    ax.set_ylim(0, 1.12)
    if ax is axes[0]:
        ax.set_ylabel("F2 Score")

handles = [
    mpatches.Patch(color=MODEL_COLORS[m], label=m) for m in MODELS_ORDER
] + [plt.Line2D([0],[0], color="gray", linestyle="--", label="Persistence")]
fig.legend(handles=handles, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("F2 Score — Test Set (Jan 2024 onward) — 95% Bootstrap CI",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "f2_all_models.png", bbox_inches="tight")
plt.show()

## 2. Per-Park Breakdown

F2 scores per park and threat for all four models. Green = high (> 0.80), red = low (< 0.40).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes_flat = axes.flatten()

for ax, model_name in zip(axes_flat, MODELS_ORDER):
    parks = list(res[model_name]["per_park_test"].keys())
    data  = np.array([
        [res[model_name]["per_park_test"][p][l]["f2"] for l in THREAT_LABELS]
        for p in parks
    ])
    park_display  = [PARK_LABELS.get(p, p) for p in parks]
    threat_display = [THREAT_SHORT[l] for l in THREAT_LABELS]

    im = ax.imshow(data, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(3))
    ax.set_xticklabels(threat_display)
    ax.set_yticks(range(len(parks)))
    ax.set_yticklabels(park_display)
    ax.set_title(f"{model_name} — Per-Park F2", fontweight="bold",
                 color=MODEL_COLORS[model_name])
    for i in range(len(parks)):
        for j in range(3):
            v = data[i, j]
            tc = "white" if v < 0.35 or v > 0.85 else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color=tc, fontsize=8, fontweight="bold")
    plt.colorbar(im, ax=ax, label="F2")

plt.tight_layout()
plt.savefig(RESULTS / "plots" / "per_park_f2_all_models.png", bbox_inches="tight")
plt.show()

In [ ]:
park_rows = []
for model_name in MODELS_ORDER:
    for park, pm in res[model_name]["per_park_test"].items():
        row = {"Model": model_name, "Park": PARK_LABELS.get(park, park)}
        for label in THREAT_LABELS:
            row[THREAT_SHORT[label]] = round(pm[label]["f2"], 3)
        park_rows.append(row)

df_park = pd.DataFrame(park_rows).set_index(["Model", "Park"])
df_park

## 3. Calibration Diagnostics

Reliability diagrams — predicted probability vs. observed event frequency.
Points on the diagonal = perfect calibration.
All models use Platt scaling on the validation set; where the Platt coefficient was
negative (vegetation in every model), temperature scaling was applied as a fallback.
Temperature scaling can only stretch or compress probabilities — it cannot invert the
signal — producing trustworthy vegetation probability estimates on the dashboard.

In [ ]:
device = torch.device("cpu")

# Tree models
_, _, test_df = load_splits()
with open(MODELS / "rf_supervised"  / "model.pkl", "rb") as f: rf_b  = pickle.load(f)
with open(MODELS / "xgb_supervised" / "model.pkl", "rb") as f: xgb_b = pickle.load(f)

X_test_flat, Y_test_flat, _ = prep_arrays(test_df, rf_b["imputer"])
rf_probs  = apply_platt(np.column_stack([p[:,1] for p in rf_b["model"].predict_proba(X_test_flat)]),  rf_b["calibrators"])
xgb_probs = apply_platt(np.column_stack([m.predict_proba(X_test_flat)[:,1] for m in xgb_b["models"]]), xgb_b["calibrators"])

# Neural models
(*_, X_test_seq, Y_test_seq, _parks) = build_sequence_splits()

with open(MODELS / "lstm_supervised" / "model.pkl", "rb") as f: lstm_b = pickle.load(f)
lstm_model = ThreatLSTM(**lstm_b["model_config"])
lstm_model.load_state_dict(lstm_b["model_state"])
lstm_probs = apply_platt(lstm_predict(lstm_model, X_test_seq, device), lstm_b["calibrators"])

with open(MODELS / "transformer_supervised" / "model.pkl", "rb") as f: tf_b = pickle.load(f)
tf_model = ThreatTransformer(**tf_b["model_config"])
tf_model.load_state_dict(tf_b["model_state"])
tf_probs = apply_platt(tf_predict(tf_model, X_test_seq, device), tf_b["calibrators"])

all_probs = {"RF": rf_probs, "XGBoost": xgb_probs, "LSTM": lstm_probs, "Transformer": tf_probs}
print("Probabilities loaded for all four models.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, (i, label) in zip(axes, enumerate(THREAT_LABELS)):
    y_true_raw = Y_test_flat[:, i].astype(float)
    valid  = ~np.isnan(y_true_raw)
    y_true = y_true_raw[valid].astype(int)

    for model_name, probs in all_probs.items():
        yp = probs[valid, i]
        frac, mean_pred = calibration_curve(y_true, yp, n_bins=10, strategy="uniform")
        ax.plot(mean_pred, frac, marker="o", markersize=3,
                color=MODEL_COLORS[model_name], label=model_name, linewidth=1.5)

    ax.plot([0,1],[0,1], "k--", linewidth=1, label="Perfect")
    ax.set_title(THREAT_SHORT[label], fontweight="bold")
    ax.set_xlabel("Mean predicted probability")
    if i == 0: ax.set_ylabel("Fraction of positives")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.legend(fontsize=8)

fig.suptitle("Reliability Diagrams — All Four Models", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "reliability_all_models.png", bbox_inches="tight")
plt.show()

## 4. Lead-Time Analysis

Mean days before each positive-event onset that the model first crosses P ≥ 0.70.

In [ ]:
lt_rows = []
for model_name in MODELS_ORDER:
    for label in THREAT_LABELS:
        lt = res[model_name]["test_metrics"][label]["lead_time_days"]
        lt_rows.append({"Model": model_name, "Threat": THREAT_SHORT[label],
                        "Lead Time (days)": round(lt, 2)})

df_lt = pd.DataFrame(lt_rows)
print(df_lt.pivot(index="Model", columns="Threat", values="Lead Time (days)").to_string())

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(3)
w = 0.20
offsets = [-1.5, -0.5, 0.5, 1.5]

for j, model_name in enumerate(MODELS_ORDER):
    vals = [res[model_name]["test_metrics"][l]["lead_time_days"] for l in THREAT_LABELS]
    bars = ax.bar(x + offsets[j]*w, vals, w,
                  color=MODEL_COLORS[model_name], alpha=0.85, label=model_name)
    for bar, v in zip(bars, vals):
        if v > 0.1:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
                    f"{v:.1f}d", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([THREAT_SHORT[l] for l in THREAT_LABELS])
ax.set_ylabel("Mean lead time (days)")
ax.set_title("Early-Warning Lead Time at P \u2265 0.70", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "lead_time_all_models.png", bbox_inches="tight")
plt.show()

## Summary

| Model | Mean F2 | Fire F2 | Drought F2 | Vegetation F2 |
|---|---|---|---|---|
| Random Forest     | **0.8565** | 0.9436 | **0.8818** | **0.7441** |
| XGBoost           | 0.8267 | 0.9355 | 0.8415 | 0.7030 |
| LSTM              | 0.7768 | 0.9384 | 0.7464 | 0.6458 |
| Transformer       | 0.6886 | **0.9407** | 0.5961 | 0.5291 |
| Persistence (RF)  | — | 0.7798 | 0.3991 | 0.4796 |

All four models beat the persistence baseline on all three threats.
Random Forest achieves the highest mean F2 and is the primary supervised baseline.

**Key findings**
- **Fire** is well-handled by all architectures (F2 > 0.93) — the temporal fire-history
  features provide a strong signal that all models exploit.
- **Tree models dominate drought and vegetation** — engineered rain-deficit and NDVI
  features carry the prediction without needing sequence memory.
- **Neural models overfit on this dataset** — LSTM stopped at epoch 72, Transformer at
  epoch 18; both show train loss well below val loss. The 6,414 training sequences are
  insufficient for the attention mechanism to learn useful patterns.
- **Vegetation Platt scaler inverts in every model** — consistent finding across all
  four architectures; root cause is the rainy-season (Jul–Dec 2023) validation window
  producing an anti-correlated calibration signal for vegetation. Temperature scaling
  was applied as a fallback (T = 10.0 for all models); unlike Platt scaling, temperature
  scaling can only stretch or compress probabilities and cannot invert the signal.
- **Random Forest is selected as the supervised baseline for the semi-supervised stage
  (Week 7)** based on highest mean F2 and most consistent per-park performance.